In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: g:\project-smart\smart-flight-hotel-agent


In [2]:
from backend.models.travel import TravelRequest

from backend.agents.travel_agent import (
    extract_travel_request,
)

# from backend.agents.travel_agent_7 import (
#     extract_travel_request,
# )

print("Phase 7 imports successful.")

Phase 7 imports successful.


In [3]:
request = TravelRequest(
    origin="London",
    destination="New York",
    departure_date="2026-10-02",
    return_date="2026-10-10",
    passengers=1,
    cabin_class="business",
    max_budget=2500,
    max_stops=1,
    hotel_min_rating=4,
)

request

TravelRequest(origin='London', destination='New York', departure_date=datetime.date(2026, 10, 2), return_date=datetime.date(2026, 10, 10), passengers=1, cabin_class='business', max_budget=2500.0, max_stops=1, hotel_min_rating=4.0)

In [4]:
from backend.agents.mcp_travel_agent import (
    plan_trip_via_mcp,
)

options = await plan_trip_via_mcp(request)

print("Number of matching options:", len(options))

for index, option in enumerate(options, start=1):

    print()
    print("=" * 60)
    print(f"OPTION {index}")
    print("=" * 60)

    print(
        "Flight:",
        option["flight"]["airline"],
        option["flight"]["flight_number"],
    )

    print(
        "Flight price:",
        option["flight_cost"],
    )

    print(
        "Stops:",
        option["flight"]["stops"],
    )

    print(
        "Hotel:",
        option["hotel"]["name"],
    )

    print(
        "Hotel rating:",
        option["hotel"]["rating"],
    )

    print(
        "Hotel cost:",
        option["hotel_cost"],
    )

    print(
        "Nights:",
        option["nights"],
    )

    print(
        "Total:",
        option["total_cost"],
    )

    print(
        "Remaining budget:",
        option["remaining_budget"],
    )

Flights received from MCP: 3
Hotels received from MCP: 2
Number of matching options: 5

OPTION 1
Flight: Example Air EA303
Flight price: 780.0
Stops: 1
Hotel: Example Central Hotel
Hotel rating: 4.2
Hotel cost: 1160.0
Nights: 8
Total: 1940.0
Remaining budget: 560.0

OPTION 2
Flight: Demo Airlines DA202
Flight price: 920.0
Stops: 0
Hotel: Example Central Hotel
Hotel rating: 4.2
Hotel cost: 1160.0
Nights: 8
Total: 2080.0
Remaining budget: 420.0

OPTION 3
Flight: Example Air EA303
Flight price: 780.0
Stops: 1
Hotel: Demo Grand Hotel
Hotel rating: 4.5
Hotel cost: 1440.0
Nights: 8
Total: 2220.0
Remaining budget: 280.0

OPTION 4
Flight: Demo Airways DA101
Flight price: 1150.0
Stops: 1
Hotel: Example Central Hotel
Hotel rating: 4.2
Hotel cost: 1160.0
Nights: 8
Total: 2310.0
Remaining budget: 190.0

OPTION 5
Flight: Demo Airlines DA202
Flight price: 920.0
Stops: 0
Hotel: Demo Grand Hotel
Hotel rating: 4.5
Hotel cost: 1440.0
Nights: 8
Total: 2360.0
Remaining budget: 140.0


In [5]:
from backend.mcp.client import (
    search_flights_via_mcp,
    search_hotels_via_mcp,
)

flight_response = await search_flights_via_mcp(
    origin=request.origin,
    destination=request.destination,
    departure_date=str(request.departure_date),
    cabin_class=request.cabin_class,
)

hotel_response = await search_hotels_via_mcp(
    destination=request.destination,
    hotel_min_rating=request.hotel_min_rating,
)

print("FLIGHT RESPONSE:")
print(flight_response)

print("\nHOTEL RESPONSE:")
print(hotel_response)

FLIGHT RESPONSE:
{'result': [{'airline': 'Demo Airways', 'flight_number': 'DA101', 'origin': 'London', 'destination': 'New York', 'departure_date': '2026-10-02', 'departure_time': '09:00', 'arrival_time': '21:30', 'cabin_class': 'business', 'price': 1150.0, 'stops': 1, 'duration_hours': 8.5}, {'airline': 'Demo Airlines', 'flight_number': 'DA202', 'origin': 'London', 'destination': 'New York', 'departure_date': '2026-10-02', 'departure_time': '12:00', 'arrival_time': '23:00', 'cabin_class': 'business', 'price': 920.0, 'stops': 0, 'duration_hours': 8.0}, {'airline': 'Example Air', 'flight_number': 'EA303', 'origin': 'London', 'destination': 'New York', 'departure_date': '2026-10-02', 'departure_time': '16:00', 'arrival_time': '04:30', 'cabin_class': 'business', 'price': 780.0, 'stops': 1, 'duration_hours': 9.5}]}

HOTEL RESPONSE:
{'result': [{'name': 'Demo Grand Hotel', 'city': 'New York', 'rating': 4.5, 'nightly_price': 180.0, 'currency': 'GBP', 'address': '123 Demo Street'}, {'name': '

In [6]:
from backend.agents.travel_agent_7 import run_phase7_travel_agent

answer = await run_phase7_travel_agent(request)

print(answer)

Flights received from MCP: 3
Hotels received from MCP: 2
**Best available option:** Example Air **EA303** with Example Central Hotel.

- Flight: **$780** — 1 stop, 9.5 hours  
- Hotel: **$1,160** — 8 nights, rated 4.2/5  
- **Total: $1,940**
- **Remaining budget: $560**

It fits your one-stop limit, meets your minimum hotel rating, and has the lowest total cost among the options. The listing doesn’t confirm the flight’s cabin class; business-class availability and included services vary by airline, route, aircraft, and fare, so please confirm with the airline before choosing. No booking has been made.


In [8]:
import inspect
from pprint import pprint

from backend.agents.mcp_travel_agent import plan_trip_via_mcp

if inspect.iscoroutinefunction(plan_trip_via_mcp):
    options = await plan_trip_via_mcp(request)
else:
    options = plan_trip_via_mcp(request)

print("Output type:", type(options).__name__)
print(
    "Number of options:",
    len(options) if isinstance(options, list) else "Not a list"
)
print("\nFirst option:")
pprint(options[0] if isinstance(options, list) and options else options)

Flights received from MCP: 3
Hotels received from MCP: 2
Output type: list
Number of options: 5

First option:
{'budget': 2500.0,
 'flight': {'airline': 'Example Air',
            'arrival_time': '04:30',
            'cabin_class': 'business',
            'departure_date': '2026-10-02',
            'departure_time': '16:00',
            'destination': 'New York',
            'duration_hours': 9.5,
            'flight_number': 'EA303',
            'origin': 'London',
            'price': 780.0,
            'stops': 1},
 'flight_cost': 780.0,
 'hotel': {'address': '45 Example Avenue',
           'city': 'New York',
           'currency': 'GBP',
           'name': 'Example Central Hotel',
           'nightly_price': 145.0,
           'rating': 4.2},
 'hotel_cost': 1160.0,
 'nights': 8,
 'remaining_budget': 560.0,
 'total_cost': 1940.0}
